# Random Forest Airline Satisfaction Analysis

In [ ]:
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split,GridSearchCV,PredefinedSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,classification_report
df=pd.read_csv('Invistico_Airline.csv')
df.head()

In [ ]:
df=df.drop_duplicates()
df=df.fillna(df.mode().iloc[0])
X=pd.get_dummies(df.drop(columns=['satisfaction']),drop_first=True)
y=df['satisfaction']
X_train,X_temp,y_train,y_temp=train_test_split(X,y,test_size=0.4,random_state=42,stratify=y)
X_val,X_test,y_val,y_test=train_test_split(X_temp,y_temp,test_size=0.5,random_state=42,stratify=y_temp)
print(X_train.shape,X_val.shape,X_test.shape)

In [ ]:
X_gs=pd.concat([X_train,X_val])
y_gs=pd.concat([y_train,y_val])
test_fold=[-1]*len(X_train)+[0]*len(X_val)
ps=PredefinedSplit(test_fold)
param_grid={'max_depth':[10,15,None],'n_estimators':[100,200],'min_samples_leaf':[1,2,4]}
rf=RandomForestClassifier(random_state=42)
grid=GridSearchCV(rf,param_grid,cv=ps,scoring='f1_weighted',n_jobs=-1)
grid.fit(X_gs,y_gs)
print(grid.best_params_)

In [ ]:
best_rf=grid.best_estimator_
pred=best_rf.predict(X_test)
print('Accuracy',accuracy_score(y_test,pred))
print('Precision',precision_score(y_test,pred,average='weighted'))
print('Recall',recall_score(y_test,pred,average='weighted'))
print('F1',f1_score(y_test,pred,average='weighted'))
dt=DecisionTreeClassifier(random_state=42)
dt.fit(X_train,y_train)
p2=dt.predict(X_test)
print('DT F1',f1_score(y_test,p2,average='weighted'))

In [ ]:
fi=pd.Series(best_rf.feature_importances_,index=X.columns).sort_values(ascending=False)
print(fi.head(10))